[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.fr/cap05/cap05.EPs_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

## 💻 **Partie pratique avec exercices de programmation**

🚧 **En construction !**

La présente liste d'exercices de programmation (EP) consolide les formulations théoriques présentées tout au long du chapitre 5 — Transformées et compression — au moyen d'un parcours pratique appliqué. Les exercices sont structurés à partir de matrices de dimensions réduites, permettant la validation analytique et l'inspection manuelle de chaque coefficient, tout en maintenant la cohérence méthodologique adoptée dans les chapitres précédents.

L'enchaînement des exercices reproduit rigoureusement le flux conceptuel du chapitre : on commence par l'implémentation explicite de la Transformée de Fourier discrète (TFD) à partir de sa définition mathématique fondamentale ; on progresse vers la conception de filtres passe-bas et de masques *notch* dans le domaine fréquentiel ; on applique la quantification des coefficients (noyau de la compression avec perte) ; et l'on conclut par l'intégration de ces étapes dans la construction d'un *pipeline* de compression JPEG simplifié ainsi que par l'analyse perceptuelle des formats d'image.

> ### ❗ Directives pour la résolution des exercices de programmation
>
> Dans tous les exercices de ce chapitre, les coordonnées du **centre du spectre** (origine des fréquences spatiales après application du décalage `fftshift`) doivent être déterminées par division entière. Pour une matrice à $L$ lignes et $C$ colonnes, la composante de fréquence nulle se situe à la position :
>
> $$
> (c_y, c_x) = \left( \left\lfloor \frac{L}{2} \right\rfloor, \left\lfloor \frac{C}{2} \right\rfloor \right)
> $$
>
> Cette convention est rigoureusement identique à celle adoptée par la fonction `np.fft.fftshift`. De plus, dans toutes les étapes exigeant une discrétisation ou un arrondi numérique (que ce soit dans la quantification des coefficients AC ou dans la reconstruction finale des pixels), on doit employer l'arrondi standard à l'entier le plus proche (*round half away from zero*), afin de limiter les ambiguïtés pour les valeurs dont la fraction est exactement égale à $0.5$.

### 🎯 Objectif de ce cahier

Ce cahier permet de développer, valider, organiser et tester des solutions d'**Exercices de Programmation (EPs)** dans des environnements interactifs, comme Colab, avec les mêmes cas de test que Moodle, en les y copiant uniquement au moment d'enregistrer la note officielle.

#### *Téléchargement*

Téléchargez `morph.py` et `testsuite.py` en exécutant la cellule ci-dessous :

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Exécution des tests
Pour évaluer les tests, exécutez `TestSuite("EP05_01.extensão").run()` dans une nouvelle cellule, en remplaçant l’extension par celle du langage utilisé (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). Le système télécharge les cas de test depuis GitHub, exécute le programme et calcule automatiquement la note.

Pour tester directement du code Python, sans sauvegarder de fichier, utilisez `run_code(codigo)` en passant le code sous forme de *chaîne de caractères* dans une variable `codigo` :

```python
codigo = """
from morph import mm
# ... votre code ici ...
"""
TestSuite("EP05_01").run_code(codigo)
```

### EP05_01 🟢 Filtre Passe-Bas Idéal par Distance dans le Spectre

Dans un ***scanner* de documents anciens**, le capteur capte le papier froissé et la texture des fibres en même temps que le texte — un bruit haute fréquence qui « pollue » le spectre sur les bords. Le technicien de maintenance n'a pas accès à l'image originale, seulement au **spectre de magnitude déjà calculé** par le logiciel du *scanner*. Son travail est simple et chirurgical : ne conserver que le **cercle central** des basses fréquences (la structure globale du document) et effacer tout ce qui se trouve hors du rayon $D_0$, éliminant la texture fine sans même avoir à toucher à l'image spatiale.

C'est le **filtre passe-bas idéal (LPFI)** : l'opération spectrale la plus directe du chapitre, mais aussi celle qui révèle le mieux l'anatomie d'un spectre centré.

#### 📋 Directives d'implémentation

1. **Dimensions :** Lire les entiers $L$ (lignes) et $C$ (colonnes) du spectre de magnitude — déjà fourni **centré** (équivalent à la sortie de `np.fft.fftshift`).
2. **Fréquence de coupure :** Lire l'entier $D_0$.
3. **Données :** Lire les valeurs entières de la matrice de magnitude, ligne par ligne.
4. **Centre du spectre :** Calculer $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
5. **Distance :** Pour chaque position $(u,v)$, calculer
$$
D(u,v) = \sqrt{(u-c_y)^2 + (v-c_x)^2}
$$
6. **Masque idéal :** Appliquer
$$
H(u,v) = \begin{cases} 1, & D(u,v) \le D_0 \\ 0, & D(u,v) > D_0 \end{cases}
$$
7. **Filtrage :** La valeur de sortie est $\text{mag}'(u,v) = \text{mag}(u,v) \cdot H(u,v)$.
8. **Sortie :** Afficher la matrice filtrée avec les dimensions $L \times C$.

#### 📌 Contraintes computationnelles

* **Comparaison non stricte :** le critère utilise $D(u,v) \le D_0$ (la frontière appartient au filtre, c'est-à-dire qu'elle est conservée).
* **Type :** toutes les valeurs d'entrée et de sortie sont des entiers ; la distance est calculée en virgule flottante uniquement en interne.
* **Pas d'arrondi de magnitude :** comme l'entrée est déjà entière et que le masque est binaire (0 ou 1), la sortie n'a jamais besoin d'arrondi.

#### 🧠 Fondement théorique

| Région | Distance au centre | Effet du filtre |
|---|---|---|
| **Centre** ($D \le D_0$) | Basses fréquences | Préservées — structure globale conservée |
| **Bords** ($D > D_0$) | Hautes fréquences | Mises à zéro — texture et bruit supprimés |
| **$D_0$ petit** | — | L'image reconstruite serait très floue |
| **$D_0$ grand** | — | Peu de filtrage ; presque toute l'énergie est préservée |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $L$.
* Ligne 2 : Entier $C$.
* Ligne 3 : Entier $D_0$.
* Lignes suivantes : Éléments entiers de la matrice de magnitude (centrée).

**Sortie :**

* Matrice filtrée en $L$ lignes et $C$ colonnes, séparés par des espaces.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 3<br>3<br>1<br>10 20 30<br>40 50 60<br>70 80 90 | 0 20 0<br>40 50 60<br>0 80 0 | Centre $(1,1)$. Les coins ont $D=\sqrt{2}\approx1.41 > 1$, donc ils sont mis à zéro ; les voisins orthogonaux ont $D=1 \le 1$ et sont conservés. |
| 1<br>3<br>0<br>5 9 7 | 0 9 0 | $L=1, C=3$ : centre en $(0,1)$. Seule la position centrale elle-même ($D=0$) survit à $D_0=0$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0501" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0501 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0501 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0501 button:hover { background: #e8dfcf; }
  #sim-ep0501 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0501_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0501_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0501_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 12px; }
  .sim-ep0501_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim-ep0501_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim-ep0501_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim-ep0501_cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP05_01 : Filtre Passe-Bas Idéal</span>
  <span class="sim-ep0501_pill">H = (D &le; D₀) ? 1 : 0</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0501_panel" style="margin-bottom:14px;">
    
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Rayon de coupure (D₀) : <span id="sim-ep0501_vl_d0" style="font-family:monospace; color:#26241d;">1</span>
      </label>
    </div>
    
    <input id="sim-ep0501_sl_d0" type="range" min="0" max="4" step="1" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajustez D₀ et observez quelles positions du spectre 5&times;5 survivent au filtre.
    </div>

  </div>

  <!-- Exibição das Grades de Espectro -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0501_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Spectre Original (Magnitude)
      </div>
      <div id="sim-ep0501_grid_orig" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0501_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Résultat Filtré
      </div>
      <div id="sim-ep0501_grid_new" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0501_debug" class="sim-ep0501_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    –
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep01(root){
    if (!root || root.dataset.sim05Ep01Init) return;
    root.dataset.sim05Ep01Init = "1";

    var N = 5, cy = Math.floor(N / 2), cx = Math.floor(N / 2);
    var mag = [];
    for (var i = 0; i < N; i++){
      var row = [];
      for (var j = 0; j < N; j++){
        row.push(10 * (i + 1) + j + 1);
      }
      mag.push(row);
    }

    var d0el = root.querySelector('#sim-ep0501_sl_d0');
    var d0v  = root.querySelector('#sim-ep0501_vl_d0');
    var go   = root.querySelector('#sim-ep0501_grid_orig');
    var gn   = root.querySelector('#sim-ep0501_grid_new');
    var dbg  = root.querySelector('#sim-ep0501_debug');

    function cellStyle(active){
      if (active) {
        return 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      } else {
        return 'background:#fafaf7; color:#8a8371; border:1px solid #e4dcc8;';
      }
    }

    function render(){
      var D0 = parseInt(d0el.value, 10);
      d0v.textContent = D0;
      go.innerHTML = '';
      gn.innerHTML = '';
      var kept = 0;

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var d = Math.sqrt((i - cy) * (i - cy) + (j - cx) * (j - cx));
          var keep = d <= D0;
          if (keep) kept++;

          var co = document.createElement('div');
          co.className = 'sim-ep0501_cell';
          co.style.cssText = cellStyle(true);
          co.textContent = mag[i][j];
          go.appendChild(co);

          var cn = document.createElement('div');
          cn.className = 'sim-ep0501_cell';
          cn.style.cssText = cellStyle(keep);
          cn.textContent = keep ? mag[i][j] : 0;
          gn.appendChild(cn);
        }
      }

      dbg.textContent = 'Centre = (' + cy + ', ' + cx + ')  |  D₀ = ' + D0 + '  |  Coeficientes mantidos: ' + kept + ' / ' + (N * N);
    }

    d0el.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep01(){
    var root = document.getElementById('sim-ep0501');
    if (root) initSim05Ep01(root); else setTimeout(tryInitSim05Ep01, 200);
  }
  tryInitSim05Ep01();
})();
</script>
""")

**Figure 5.1:** Simulateur EP05_01 : Filtre passe-bas idéal dans le spectre


<figure id="fig-05-sim-ep0501">
  <img src="imagens/fig-05-sim-ep0501.png" alt=" Simulateur EP05_01 : Filtre passe-bas idéal dans le spectre " style="max-width:80%" />
  <figcaption><strong>Figure 5.1:</strong>  Simulateur EP05_01 : Filtre passe-bas idéal dans le spectre </figcaption>
</figure>

In [ ]:
%%writefile EP05_01.py
# Code Python

In [ ]:
TestSuite("EP05_01.py").run()

### EP05_02 🟡 Filtre *Notch* : Suppression des pics périodiques

Une caméra d'**inspection industrielle** capture des images de circuits imprimés, mais l'alimentation électrique de la ligne de production introduit une **interférence électrique périodique** — un motif de stries quasi imperceptible à l'œil nu, mais qui apparaît dans le spectre de Fourier comme des **paires de pics brillants** symétriquement positionnés autour du centre. L'équipe de vision par ordinateur ne peut pas recapturer l'image : elle doit **localiser et effacer chirurgicalement** ces paires de pics dans le spectre, tout en préservant le reste de l'information utile de l'image.

C'est le rôle du **filtre coupe-bande *notch*** : contrairement au passe-bas (qui affecte une région continue), il cible **des points spécifiques et leurs symétriques**, laissant le reste du spectre intact.

#### 📋 Directives d'implémentation

1. **Dimensions :** Lire les entiers $L$ (lignes) et $C$ (colonnes) du spectre de magnitude centré.
2. **Données :** Lire les valeurs entières de la matrice de magnitude, ligne par ligne.
3. **Pics :** Lire l'entier $K$ (nombre de paires de pics à supprimer).
4. **Pour chacun des $K$ pics :** lire trois entiers $\Delta v$, $\Delta u$, $r$ — déplacement vertical, déplacement horizontal et rayon du *notch*.
5. **Centre du spectre :** $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
6. **Suppression symétrique :** pour chaque pic, mettre à zéro **toutes** les positions $(u,v)$ telles que la distance au point $(c_y+\Delta v,\, c_x+\Delta u)$ soit $\le r$, **et également** toutes les positions à distance $\le r$ du point symétrique $(c_y-\Delta v,\, c_x-\Delta u)$.
7. **Sortie :** Afficher la matrice résultante avec les dimensions $L \times C$.

#### 📌 Contraintes computationnelles

* **Symétrie obligatoire :** chaque pic fourni génère **deux** disques mis à zéro (le point et son symétrique par rapport au centre) — oublier le symétrique est l'erreur la plus courante.
* **Chevauchement :** si deux disques se chevauchent, la position reste à zéro (ni « addition » ni restauration).
* **Comparaison non stricte :** une position est mise à zéro si $\text{distance} \le r$.
* **Ordre de lecture :** les $K$ pics doivent être traités dans l'ordre où ils apparaissent en entrée, mais le résultat final est indépendant de l'ordre (les opérations de mise à zéro sont commutatives).

#### 🧠 Fondement théorique

| Concept | Rôle dans le filtre *notch* |
|---|---|
| **Pic en $(\Delta v, \Delta u)$** | Fréquence de l'interférence périodique détectée visuellement dans le spectre |
| **Point symétrique $(-\Delta v,-\Delta u)$** | Toute DFT d'un signal réel est hermitienne : les pics apparaissent toujours en paires symétriques par rapport au centre |
| **Rayon $r$** | Contrôle la « largeur » de la réjection — un $r$ grand élimine davantage d'énergie autour du pic, mais aussi davantage d'information utile |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $L$.
* Ligne 2 : Entier $C$.
* Lignes suivantes : Éléments entiers de la matrice de magnitude (centrée), $L$ lignes.
* Ligne suivante : Entier $K$.
* $K$ lignes suivantes : trois entiers $\Delta v$, $\Delta u$, $r$ (séparés par des espaces).

**Sortie :**

* Matrice résultante sur $L$ lignes et $C$ colonnes, séparés par des espaces.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 5<br>5<br>1 2 3 4 5<br>6 7 8 9 10<br>11 12 13 14 15<br>16 17 18 19 20<br>21 22 23 24 25<br>1<br>1 1 0 | 1 2 3 4 5<br>6 0 8 9 10<br>11 12 13 14 15<br>16 17 18 0 20<br>21 22 23 24 25 | Centre $(c_y, c_x) = (2, 2)$. Pic fourni $(\Delta v, \Delta u) = (1, 1)$ génère le point $(3, 3)$ (valeur 19) et son symétrique $(1, 1)$ (valeur 7), tous deux mis à zéro avec $r=0$ (seuls les points exacts). |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0502" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0502 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0502 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0502 button:hover { background: #e8dfcf; }
  #sim-ep0502 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0502_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0502_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0502_grid_ctrls { display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 12px; }
  .sim-ep0502_cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP05_02 : Filtre Notch</span>
  <span class="sim-ep0502_pill">Paire symétrique</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0502_panel" style="margin-bottom:14px;">
    <div class="sim-ep0502_grid_ctrls">
      
      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">&Delta;v</label>
          <span id="sim-ep0502_vl_dv" style="font-family:monospace; font-weight:700; color:#26241d;">1</span>
        </div>
        <input id="sim-ep0502_sl_dv" type="range" min="-2" max="2" step="1" value="1">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">&Delta;u</label>
          <span id="sim-ep0502_vl_du" style="font-family:monospace; font-weight:700; color:#26241d;">1</span>
        </div>
        <input id="sim-ep0502_sl_du" type="range" min="-2" max="2" step="1" value="1">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Rayon (r)</label>
          <span id="sim-ep0502_vl_r" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0502_sl_r" type="range" min="0" max="2" step="1" value="0">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:10px; text-align:center;">
      Déplacez &Delta;v e &Delta;u pour choisir le pic &mdash; observez que la paire symétrique est également filtrée.
    </div>
  </div>

  <!-- Espectro 5x5 -->
  <div class="sim-ep0502_panel" style="text-align:center; margin-bottom:14px;">
    <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
      Spectre 5&times;5 (Rouge = Supprimé par le filtre)
    </div>
    <div id="sim-ep0502_grid" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0502_debug" class="sim-ep0502_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep02(root){
    if (!root || root.dataset.sim05Ep02Init) return;
    root.dataset.sim05Ep02Init = "1";

    var N = 5, cy = Math.floor(N / 2), cx = Math.floor(N / 2);
    var mag = [];
    for (var i = 0; i < N; i++){
      var row = [];
      for (var j = 0; j < N; j++){
        row.push(i * 5 + j + 1);
      }
      mag.push(row);
    }

    var dv  = root.querySelector('#sim-ep0502_sl_dv');
    var du  = root.querySelector('#sim-ep0502_sl_du');
    var r   = root.querySelector('#sim-ep0502_sl_r');
    var dvv = root.querySelector('#sim-ep0502_vl_dv');
    var duv = root.querySelector('#sim-ep0502_vl_du');
    var rv  = root.querySelector('#sim-ep0502_vl_r');

    var grid = root.querySelector('#sim-ep0502_grid');
    var dbg  = root.querySelector('#sim-ep0502_debug');

    function cellStyle(kill){
      if (kill) {
        return 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
      } else {
        return 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      }
    }

    function render(){
      var DV = parseInt(dv.value, 10);
      var DU = parseInt(du.value, 10);
      var R  = parseInt(r.value, 10);

      dvv.textContent = DV;
      duv.textContent = DU;
      rv.textContent  = R;

      var p1 = [cy + DV, cx + DU];
      var p2 = [cy - DV, cx - DU];

      grid.innerHTML = '';
      var removed = 0;

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var d1 = Math.sqrt((i - p1[0]) * (i - p1[0]) + (j - p1[1]) * (j - p1[1]));
          var d2 = Math.sqrt((i - p2[0]) * (i - p2[0]) + (j - p2[1]) * (j - p2[1]));
          var kill = (d1 <= R) || (d2 <= R);

          if (kill) removed++;

          var c = document.createElement('div');
          c.className = 'sim-ep0502_cell';
          c.style.cssText = cellStyle(kill);
          c.textContent = kill ? 0 : mag[i][j];
          grid.appendChild(c);
        }
      }

      dbg.textContent = 'Centre = (' + cy + ', ' + cx + ')  |  Pico = (' + p1[0] + ', ' + p1[1] + ')  |  Simétrico = (' + p2[0] + ', ' + p2[1] + ')  |  Removidos: ' + removed;
    }

    [dv, du, r].forEach(function(el){
      el.addEventListener('input', render);
    });

    render();
  }

  function tryInitSim05Ep02(){
    var root = document.getElementById('sim-ep0502');
    if (root) initSim05Ep02(root); else setTimeout(tryInitSim05Ep02, 200);
  }
  tryInitSim05Ep02();
})();
</script>
""")

**Figure 5.2:** Simulateur EP05_02: Filtre Notch


<figure id="fig-05-sim-ep0502">
  <img src="imagens/fig-05-sim-ep0502.png" alt=" Simulateur EP05_02: Filtre Notch " style="max-width:80%" />
  <figcaption><strong>Figure 5.2:</strong>  Simulateur EP05_02: Filtre Notch </figcaption>
</figure>

In [ ]:
%%writefile EP05_02.py
# Code Python

In [ ]:
TestSuite("EP05_02.py").run()

### EP05_03 🟠 Quantification DCT : la véritable source de compression

Une application de **galerie de photos** doit réduire la taille de milliers d'images avant de les *téléverser* vers le cloud, sans tout recoder de zéro. L'ingénieur responsable dispose déjà des **coefficients DCT** de chaque bloc $4\times4$ calculés (l'étape coûteuse en calcul est déjà faite) — il ne reste plus qu'à appliquer la **table de quantification**, l'étape qui élimine réellement de l'information et génère la compression. Les coefficients haute fréquence, moins perceptibles à l'œil humain, reçoivent de grands diviseurs et tendent à devenir **zéro** ; les coefficients basse fréquence, plus perceptibles, reçoivent de petits diviseurs et survivent presque intacts.

Vous allez implémenter exactement cette étape : **quantifier et déquantifier** (diviser, arrondir, multiplier en retour) — le cœur de la compression *lossy* du JPEG.

#### 📋 Directives d'implémentation

1. **Dimension du bloc :** Lire l'entier $N$ (bloc $N \times N$).
2. **Coefficients :** Lire la matrice $C$ des coefficients DCT, $N$ lignes avec $N$ entiers chacune (ils peuvent être négatifs).
3. **Table de quantification :** Lire la matrice $Q$, $N$ lignes avec $N$ entiers positifs chacune.
4. **Quantification :** Pour chaque position $(u,v)$, calculer l'indice quantifié
$$
\tilde{C}(u,v) = \text{round}\!\left(\frac{C(u,v)}{Q(u,v)}\right)
$$
en utilisant l'arrondi standard à l'entier le plus proche (les valeurs intermédiaires `.5` ne se produisent jamais dans les cas de test).
5. **Déquantification (reconstruction) :** Calculer
$$
C'(u,v) = \tilde{C}(u,v) \times Q(u,v)
$$
6. **Sortie :** Afficher la matrice reconstruite $C'$, $N \times N$, entiers.

#### 📌 Contraintes informatiques

* ***Aller-retour* complet :** la sortie est le coefficient **reconstruit** ($\tilde{C} \times Q$), pas l'indice quantifié isolé.
* **Division en virgule flottante :** la division $C(u,v)/Q(u,v)$ doit être effectuée en virgule flottante avant l'arrondi — une division entière tronquée produirait un résultat incorrect.
* **Signe préservé :** les coefficients négatifs conservent leur signe après quantification et reconstruction.
* **$Q(u,v) > 0$ toujours :** aucune gestion de division par zéro n'est nécessaire.

#### 🧠 Fondements théoriques

| Coefficient | Fréquence | Valeur typique de $Q$ | Effet de la quantification |
|---|---|---|---|
| $C(0,0)$ | DC (moyenne du bloc) | Petite | Survit presque toujours — domine l'énergie |
| $C(u,v)$ faible $u+v$ | Basse fréquence | Petite/moyenne | Partiellement préservé |
| $C(u,v)$ élevé $u+v$ | Haute fréquence | Grande | Devient souvent zéro — source de la compression |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $N$.
* $N$ lignes suivantes : matrice $C$ (coefficients DCT, entiers, peuvent être négatifs).
* $N$ lignes suivantes : matrice $Q$ (table de quantification, entiers positifs).

**Sortie :**

* Matrice reconstruite $C'$, $N \times N$, entiers séparés par des espaces.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 4<br>50 10 -5 0<br>8 -3 2 1<br>0 1 0 0<br>2 0 0 -1<br>2 5 7 8<br>4 7 8 11<br>6 8 11 12<br>9 11 12 14 | 50 10 -7 0<br>8 0 0 0<br>0 0 0 0<br>0 0 0 0 | $C(0,0)=50/2=25 \to 25\times2=50$ (préservé). $C(0,2)=-5/7\approx-0.71\to-1\to-1\times7=-7$. Quant à $C(1,1)=-3/7\approx-0.43\to0$ : mis à zéro par la quantification — la majeure partie du bloc devient zéro, illustrant la compaction de l'énergie dans le coin supérieur gauche. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0503" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0503 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0503 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0503 button:hover { background: #e8dfcf; }
  #sim-ep0503 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0503_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0503_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0503_cell { width: 46px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP05_03 : Quantification DCT</span>
  <span class="sim-ep0503_pill">round(C / Q) &times; Q</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0503_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Échelle de Q (Agressivité) : <span id="sim-ep0503_vl_s" style="font-family:monospace; color:#26241d;">1.00&times;</span>
      </label>
    </div>
    
    <input id="sim-ep0503_sl_s" type="range" min="0.25" max="4" step="0.25" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajustez l'échelle de Q et voyez combien de coefficients survivent (non nuls) après l'aller-retour.
    </div>
  </div>

  <!-- Exibição das Grades 4x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0503_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Coefficients DCT (C)
      </div>
      <div id="sim-ep0503_grid_c" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0503_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Reconstruit (round(C / Q) &middot; Q)
      </div>
      <div id="sim-ep0503_grid_r" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0503_debug" class="sim-ep0503_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep03(root){
    if (!root || root.dataset.sim05Ep03Init) return;
    root.dataset.sim05Ep03Init = "1";

    var C = [[50, 10, -5, 0], [8, -3, 2, 1], [0, 1, 0, 0], [2, 0, 0, -1]];
    var Qbase = [[2, 5, 7, 8], [4, 7, 8, 11], [6, 8, 11, 12], [9, 11, 12, 14]];

    var s   = root.querySelector('#sim-ep0503_sl_s');
    var sv  = root.querySelector('#sim-ep0503_vl_s');
    var gc  = root.querySelector('#sim-ep0503_grid_c');
    var gr  = root.querySelector('#sim-ep0503_grid_r');
    var dbg = root.querySelector('#sim-ep0503_debug');

    function cell(v, faded){
      var c = document.createElement('div');
      c.className = 'sim-ep0503_cell';
      if (faded) {
        c.style.cssText = 'background:#fafaf7; color:#8a8371; border:1px solid #e4dcc8;';
      } else {
        c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      }
      c.textContent = v;
      return c;
    }

    function render(){
      var scale = parseFloat(s.value);
      sv.innerHTML = scale.toFixed(2) + '&times;';
      gc.innerHTML = '';
      gr.innerHTML = '';
      var zeros = 0, total = 16;

      for (var i = 0; i < 4; i++){
        for (var j = 0; j < 4; j++){
          gc.appendChild(cell(C[i][j], false));
          var Q = Qbase[i][j] * scale;
          var q = Math.round(C[i][j] / Q);
          var rec = Math.round(q * Q);
          if (rec === 0) zeros++;
          gr.appendChild(cell(rec, rec === 0));
        }
      }

      dbg.textContent = 'Zéros : ' + zeros + ' / ' + total + '  |  Quanto maior a escala de Q, mais zeros — maior compressão, menor qualidade.';
    }

    s.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep03(){
    var root = document.getElementById('sim-ep0503');
    if (root) initSim05Ep03(root); else setTimeout(tryInitSim05Ep03, 200);
  }
  tryInitSim05Ep03();
})();
</script>
""")

**Figure 5.3:** Simulateur EP05_03 : Quantification DCT (*round-trip*)


<figure id="fig-05-sim-ep0503">
  <img src="imagens/fig-05-sim-ep0503.png" alt=" Simulateur EP05_03 : Quantification DCT (*round-trip*) " style="max-width:80%" />
  <figcaption><strong>Figure 5.3:</strong>  Simulateur EP05_03 : Quantification DCT (*round-trip*) </figcaption>
</figure>

In [ ]:
%%writefile EP05_03.py
# Code Python

In [ ]:
TestSuite("EP05_03.py").run()

### EP05_04 🔴 Implémentation de la TFD 2D à partir de la définition

Un laboratoire de recherche en **astronomie computationnelle** a reçu, d'une mission ancienne, un petit capteur expérimental dont les données brutes ne peuvent pas être traitées par les bibliothèques modernes de FFT — l'environnement de validation est isolé et ne permet que des opérations arithmétiques de base. L'équipe doit **réimplémenter la Transformée de Fourier Discrète 2D à partir de la définition mathématique elle-même**, cellule par cellule, pour ensuite comparer bit à bit avec `np.fft.fft2` dans un autre environnement.

C'est l'exercice le plus conceptuel de la liste : il n'y a pas de raccourcis. Vous allez implémenter la double sommation de la [Équation 5](#eq-05-dft) directement, en mettant en évidence *pourquoi* la FFT existe — et le coût computationnel qu'elle évite.

#### 📋 Directives d'implémentation

1. **Dimensions :** Lire les entiers $M$ (lignes) et $N$ (colonnes) de l'image $f(x,y)$.
2. **Données :** Lire les valeurs entières de $f(x,y)$, ligne par ligne.
3. **TFD 2D :** Pour chaque paire de fréquences $(u,v)$ avec $u=0,\ldots,M-1$ et $v=0,\ldots,N-1$, calculer
$$
F(u,v) = \sum_{x=0}^{M-1}\sum_{y=0}^{N-1} f(x,y)\, e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$
en utilisant l'identité d'Euler $e^{-j\theta} = \cos(\theta) - j\sin(\theta)$ pour séparer les parties réelle et imaginaire — **n'utilisez aucune fonction de FFT prête à l'emploi**.
4. **Magnitude :** Calculer $|F(u,v)| = \sqrt{\text{Re}(F)^2 + \text{Im}(F)^2}$ et arrondir à l'entier le plus proche.
5. **Sortie :** Afficher la matrice des magnitudes arrondies, $M \times N$, dans le même ordre (sans `fftshift` — la composante DC reste en $(0,0)$).

#### 📌 Contraintes computationnelles

* **Interdiction d'utiliser des bibliothèques de FFT :** l'implémentation doit calculer les double sommations explicitement (boucles imbriquées), même si c'est plus lent.
* **Sans `fftshift` :** la sortie conserve la convention brute de la TFD, avec la composante DC en $F(0,0)$ (coin supérieur gauche).
* **Arrondi :** la magnitude finale doit être arrondie à l'entier le plus proche ; dans les cas de test, il n'y a pas d'ambiguïté `.5`.
* **Précision :** de petites erreurs de virgule flottante (de l'ordre de $10^{-6}$) avant l'arrondi sont attendues et n'affectent pas le résultat entier final.

#### 🧠 Fondement théorique

| Élément | Signification |
|---|---|
| $F(0,0)$ | Composante DC — somme de tous les pixels, $F(0,0) = \sum f(x,y)$ |
| Partie réelle $\text{Re}(F)$ | Projection du signal sur les cosinus |
| Partie imaginaire $\text{Im}(F)$ | Projection du signal sur les sinus |
| Complexité de cette implémentation | $\mathcal{O}((MN)^2)$ — c'est pourquoi la FFT, avec $\mathcal{O}(MN\log(MN))$, est indispensable pour les images réelles |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $M$.
* Ligne 2 : Entier $N$.
* Lignes suivantes : Éléments entiers de $f(x,y)$, $M$ lignes.

**Sortie :**

* Matrice des magnitudes $|F(u,v)|$ arrondies, $M \times N$, séparées par des espaces.

#### 📌 Exemples

| Entrée | Sortie | Remarque |
|---|---|---|
| 2<br>2<br>1 2<br>3 4 | 10 2<br>4 0 | $F(0,0)=1+2+3+4=10$ (DC = somme totale). $F(0,1)=(1-2)+(3-4)=-2 \to |F|=2$. $F(1,0)=(1+2)-(3+4)=-4\to|F|=4$. $F(1,1)=(1-2)-(3-4)=0$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0504" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0504 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0504 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0504 button:hover { background: #e8dfcf; }
  .sim-ep0504_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0504_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0504_cell { width: 52px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 13px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP05_04 : DFT 2D &mdash; Définition directe</span>
  <span class="sim-ep0504_pill">&Sigma;&Sigma; f(x,y) e<sup>-j2&pi;(&hellip;)</sup></span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Instrução -->
  <div class="sim-ep0504_panel" style="margin-bottom:14px; text-align:center;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600;">
      Cliquez sur les cellules de f(x,y) pour modifier les valeurs (incrément +1 ; Maj + clic décrément -1) et voyez |F(u,v)| recalculé en direct.
    </div>
  </div>

  <!-- Exibição das Grades 2x2 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0504_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        f(x,y) &mdash; Domaine spatial
      </div>
      <div id="sim-ep0504_grid_f" style="display:grid; grid-template-columns:repeat(2, 52px); gap:6px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0504_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        |F(u,v)| &mdash; Magnitude (sans décalage)
      </div>
      <div id="sim-ep0504_grid_F" style="display:grid; grid-template-columns:repeat(2, 52px); gap:6px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0504_debug" class="sim-ep0504_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep04(root){
    if (!root || root.dataset.sim05Ep04Init) return;
    root.dataset.sim05Ep04Init = "1";

    var f = [[1, 2], [3, 4]];
    var gf  = root.querySelector('#sim-ep0504_grid_f');
    var gF  = root.querySelector('#sim-ep0504_grid_F');
    var dbg = root.querySelector('#sim-ep0504_debug');

    function render(){
      gf.innerHTML = '';
      gF.innerHTML = '';

      for (var x = 0; x < 2; x++){
        for (var y = 0; y < 2; y++){
          (function(xx, yy){
            var c = document.createElement('div');
            c.className = 'sim-ep0504_cell';
            c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7; cursor:pointer;';
            c.textContent = f[xx][yy];
            c.addEventListener('click', function(e){
              if (e.shiftKey){ f[xx][yy]--; } else { f[xx][yy]++; }
              render();
            });
            gf.appendChild(c);
          })(x, y);
        }
      }

      var M = 2, N = 2;
      for (var u = 0; u < M; u++){
        for (var v = 0; v < N; v++){
          var re = 0, im = 0;
          for (var x = 0; x < M; x++){
            for (var y = 0; y < N; y++){
              var theta = 2 * Math.PI * (u * x / M + v * y / N);
              re += f[x][y] * Math.cos(theta);
              im -= f[x][y] * Math.sin(theta);
            }
          }
          var mag = Math.round(Math.sqrt(re * re + im * im));
          var c = document.createElement('div');
          c.className = 'sim-ep0504_cell';
          c.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          c.textContent = mag;
          gF.appendChild(c);
        }
      }

      dbg.textContent = 'F(0,0) = somme de tous les pixels = ' + (f[0][0] + f[0][1] + f[1][0] + f[1][1]) + ' (componente DC)';
    }

    render();
  }

  function tryInitSim05Ep04(){
    var root = document.getElementById('sim-ep0504');
    if (root) initSim05Ep04(root); else setTimeout(tryInitSim05Ep04, 200);
  }
  tryInitSim05Ep04();
})();
</script>
""")

**Figure 5.4:** Simulateur EP05_04: DFT 2D manuel


<figure id="fig-05-sim-ep0504">
  <img src="imagens/fig-05-sim-ep0504.png" alt=" Simulateur EP05_04: DFT 2D manuel " style="max-width:80%" />
  <figcaption><strong>Figure 5.4:</strong>  Simulateur EP05_04: DFT 2D manuel </figcaption>
</figure>

In [ ]:
%%writefile EP05_04.py
# Code Python

In [ ]:
TestSuite("EP05_04.py").run()

### EP05_05 🏆 *Pipeline* JPEG complet : DCT, quantification et reconstruction

Vous avez été chargé de créer, à partir de zéro, un **codec JPEG didactique** dans un environnement embarqué, sans aucune bibliothèque d'image disponible — uniquement des opérations mathématiques de base. Le client veut comprendre exactement où la qualité est perdue et où elle est récupérée, bloc par bloc. C'est le défi final du chapitre : intégrer **tout** ce qui a été étudié — la DCT-II orthonormale, la quantification perceptuelle et la reconstruction via IDCT — dans un seul *pipeline* de bout en bout, traitant un bloc $N \times N$ du début à la fin, exactement comme le fait le standard JPEG en interne, $8\times8$ pixels à la fois.

#### 📋 Directives d'implémentation

1. **Dimension du bloc :** Lire l'entier $N$.
2. **Bloc original :** Lire la matrice de pixels $f(x,y)$, $N$ lignes avec $N$ entiers dans $[0,255]$.
3. **Table de quantification :** Lire la matrice $Q$, $N \times N$ entiers positifs.
4. **Centrage :** Soustraire 128 de chaque pixel : $g(x,y) = f(x,y) - 128$.
5. **DCT-II 2D orthonormale :** Calculer
$$
C(u,v) = \alpha(u)\,\alpha(v)\sum_{x=0}^{N-1}\sum_{y=0}^{N-1} g(x,y)\,\cos\!\left[\frac{\pi(2x+1)u}{2N}\right]\cos\!\left[\frac{\pi(2y+1)v}{2N}\right]
$$
avec $\alpha(0)=\sqrt{1/N}$ et $\alpha(k)=\sqrt{2/N}$ pour $k>0$.
6. **Quantification :** $\tilde{C}(u,v) = \text{round}(C(u,v)/Q(u,v))$.
7. **Déquantification :** $C'(u,v) = \tilde{C}(u,v)\times Q(u,v)$.
8. **IDCT-II 2D (inverse orthonormale) :** Calculer $g'(x,y)$ à partir de $C'(u,v)$ en utilisant la transformée inverse correspondante (même base, somme sur $u,v$).
9. **Inversion du centrage et arrondi :** $f'(x,y) = \text{round}(g'(x,y) + 128)$, restreint à l'intervalle $[0,255]$ (*clipping*).
10. **Sortie :** Afficher le bloc reconstruit $f'$, $N \times N$, entiers.

#### 📌 Contraintes computationnelles

* ***Pipeline* complet obligatoire :** toutes les six étapes (centrer, DCT, quantifier, déquantifier, IDCT, inverser) doivent être implémentées — sauter la quantification ne réussit pas les tests, car le résultat serait identique à l'original.
* ***Clipping* :** les valeurs reconstruites hors de $[0,255]$ doivent être tronquées (0 si négatif, 255 si supérieur à 255).
* **Arrondi :** à la fois dans la quantification et dans la reconstruction finale des pixels, utilisez un arrondi standard ; les cas de test évitent toute ambiguïté `.5`.
* **Base orthonormale :** la normalisation $\alpha(u)$ et $\alpha(v)$ doit être appliquée exactement comme spécifié — sans elle, l'IDCT ne reconstruit pas correctement.

#### 🧠 Fondement théorique

| Étape | Analogue dans le standard JPEG réel | Où la qualité est perdue |
|---|---|---|
| Centrage | Identique — la DCT suppose un signal centré sur zéro | Aucune perte |
| DCT-II | Étapes 3–4 du *pipeline* ([Tableau 5](#tbl-05-pipeline-jpeg)) | Aucune perte (transformation exacte et réversible) |
| Quantification | Étape 5 — division par $Q(u,v)$ | **Principale source de perte** — les coefficients de haute fréquence deviennent zéro |
| IDCT | Reconstruction finale | Reconstruit exactement les coefficients *quantifiés*, pas les originaux |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $N$.
* $N$ lignes suivantes : bloc original $f(x,y)$, entiers dans $[0,255]$.
* $N$ lignes suivantes : table de quantification $Q$, entiers positifs.

**Sortie :**

* Bloc reconstruit $f'(x,y)$, $N \times N$, entiers dans $[0,255]$, séparés par des espaces.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 4<br>120 130 125 128<br>115 140 135 122<br>118 150 160 130<br>110 120 145 138<br>4 6 8 10<br>6 8 10 12<br>8 10 12 16<br>10 12 16 20 | 118 126 119 131<br>114 143 140 119<br>117 149 159 130<br>107 121 146 139 | Après DCT, quantification agressive des hautes fréquences (grandes valeurs de $Q$ en bas à droite) et reconstruction via IDCT, le bloc reste **proche** de l'original, mais pas identique — la différence est le coût de la compression *lossy*. |

#### 💡 Conseil de débogage

Si le résultat ne correspond pas, vérifiez dans cet ordre : (1) les coefficients DCT bruts (avant quantification) — ils doivent reconstruire l'original **exactement** via IDCT si vous sautez les étapes 6–7 ; (2) la table $\alpha(u)$ — erreur courante : appliquer $\sqrt{2/N}$ aussi pour $u=0$ ; (3) l'arrondi de la quantification, qui doit se produire **avant** de multiplier à nouveau par $Q$.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0505" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0505 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0505 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0505 button:hover { background: #e8dfcf; }
  #sim-ep0505 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0505_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0505_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0505_cell { width: 46px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP05_05 : Pipeline JPEG (Bloc 4&times;4)</span>
  <span class="sim-ep0505_pill">DCT &rarr; Q &rarr; IDCT</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0505_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Échelle de Q (1 = Table de base, Plus grand = Plus de perte) : <span id="sim-ep0505_vl_s" style="font-family:monospace; color:#26241d;">1.00&times;</span>
      </label>
    </div>
    
    <input id="sim-ep0505_sl_s" type="range" min="0.5" max="5" step="0.5" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajustez le facteur d'échelle de quantisation et observez le bloc reconstruit s'éloigner (ou se rapprocher) de l'original.
    </div>
  </div>

  <!-- Exibição das Grades 4x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0505_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Bloc original
      </div>
      <div id="sim-ep0505_grid_o" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0505_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Reconstruit (DCT &rarr; Q &rarr; IDCT)
      </div>
      <div id="sim-ep0505_grid_r" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0505_debug" class="sim-ep0505_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep05(root){
    if (!root || root.dataset.sim05Ep05Init) return;
    root.dataset.sim05Ep05Init = "1";

    var N = 4;
    var f = [[120, 130, 125, 128], [115, 140, 135, 122], [118, 150, 160, 130], [110, 120, 145, 138]];
    var Qbase = [[4, 6, 8, 10], [6, 8, 10, 12], [8, 10, 12, 16], [10, 12, 16, 20]];

    var s   = root.querySelector('#sim-ep0505_sl_s');
    var sv  = root.querySelector('#sim-ep0505_vl_s');
    var go  = root.querySelector('#sim-ep0505_grid_o');
    var gr  = root.querySelector('#sim-ep0505_grid_r');
    var dbg = root.querySelector('#sim-ep0505_debug');

    function alpha(k){ return k === 0 ? Math.sqrt(1 / N) : Math.sqrt(2 / N); }

    function dct2(g){
      var C = [];
      for (var u = 0; u < N; u++){ C.push(new Array(N).fill(0)); }
      for (var u = 0; u < N; u++){
        for (var v = 0; v < N; v++){
          var sum = 0;
          for (var x = 0; x < N; x++){
            for (var y = 0; y < N; y++){
              sum += g[x][y] * Math.cos(Math.PI * (2 * x + 1) * u / (2 * N)) * Math.cos(Math.PI * (2 * y + 1) * v / (2 * N));
            }
          }
          C[u][v] = alpha(u) * alpha(v) * sum;
        }
      }
      return C;
    }

    function idct2(C){
      var g = [];
      for (var x = 0; x < N; x++){ g.push(new Array(N).fill(0)); }
      for (var x = 0; x < N; x++){
        for (var y = 0; y < N; y++){
          var sum = 0;
          for (var u = 0; u < N; u++){
            for (var v = 0; v < N; v++){
              sum += alpha(u) * alpha(v) * C[u][v] * Math.cos(Math.PI * (2 * x + 1) * u / (2 * N)) * Math.cos(Math.PI * (2 * y + 1) * v / (2 * N));
            }
          }
          g[x][y] = sum;
        }
      }
      return g;
    }

    function cell(v){
      var c = document.createElement('div');
      c.className = 'sim-ep0505_cell';
      c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      c.textContent = v;
      return c;
    }

    function render(){
      var scale = parseFloat(s.value);
      sv.innerHTML = scale.toFixed(2) + '&times;';
      go.innerHTML = '';
      gr.innerHTML = '';

      var g = [];
      for (var x = 0; x < N; x++){
        var row = [];
        for (var y = 0; y < N; y++){
          row.push(f[x][y] - 128);
        }
        g.push(row);
      }

      var C = dct2(g);
      var Cq = [];
      for (var u = 0; u < N; u++){
        var row = [];
        for (var v = 0; v < N; v++){
          var Qv = Qbase[u][v] * scale;
          var q = Math.round(C[u][v] / Qv);
          row.push(q * Qv);
        }
        Cq.push(row);
      }

      var gr2 = idct2(Cq);
      var diffSum = 0, n = 0;

      for (var x = 0; x < N; x++){
        for (var y = 0; y < N; y++){
          go.appendChild(cell(f[x][y]));
          var rec = Math.round(gr2[x][y] + 128);
          rec = Math.max(0, Math.min(255, rec));
          gr.appendChild(cell(rec));
          diffSum += Math.abs(rec - f[x][y]);
          n++;
        }
      }

      dbg.textContent = 'Erreur moyenne absolue par pixel : ' + (diffSum / n).toFixed(2) + '  |  Quanto maior a escala de Q, maior o erro de reconstrução.';
    }

    s.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep05(){
    var root = document.getElementById('sim-ep0505');
    if (root) initSim05Ep05(root); else setTimeout(tryInitSim05Ep05, 200);
  }
  tryInitSim05Ep05();
})();
</script>
""")

**Figure 5.5:** Simulateur EP05_05 : *Pipeline* JPEG complet par bloc


<figure id="fig-05-sim-ep0505">
  <img src="imagens/fig-05-sim-ep0505.png" alt=" Simulateur EP05_05 : *Pipeline* JPEG complet par bloc " style="max-width:80%" />
  <figcaption><strong>Figure 5.5:</strong>  Simulateur EP05_05 : *Pipeline* JPEG complet par bloc </figcaption>
</figure>

In [ ]:
%%writefile EP05_05.py
# Code Python

In [ ]:
TestSuite("EP05_05.py").run()